# Credit Card Churn - Prediction
---

### CRISP-DM Methodology
This project follows the CRISP-DM (*Cross-Industry Standard Process for Data Mining*) framework applied to **Customer Retention & Churn Prediction**:
| **Stage** | **Objective** | **Methodological Execution** |
| :--- | :--- | :--- |
| **1. Business Understanding** | Mitigate revenue loss by identifying at-risk customers. | • **Target Definition**: Binary Classification (Churn: Yes/No).<br>• **KPIs**: Maximize **Lift** in retention campaigns & Revenue Saved vs. Cost. |
| **2. Data Understanding** | Detect patterns of friction and dissatisfaction. | • **EDA**: Distribution analysis (Detect Imbalance).<br>• **Hypothesis Testing**: Correlation Matrix & Independence Tests (Chi-Square). |
| **3. Data Preparation** | Construct a robust dataset for parametric modeling. | • **Scaling**: Standardization (Z-score) for coefficient comparability.<br>• **Encoding**: One-Hot Encoding for nominal variables.<br>• **Splitting**: Stratified Train/Test Split to preserve class ratio. |
| **4. Modeling** | Estimate Churn Probability | • **Algorithms**: Logistic Regression, LinearSVC, KNN, Random Florest, XGBoost, LightGBM.<br>• **Inference**: Analyze **Odds Ratios** to determine feature elasticity. |
| **5. Evaluation** | Assess model reliability and financial impact. | • **Discrimination**: AUC-ROC & F1-Score (Threshold Tuning).<br>• **Calibration**: Probability Calibration Curve (Reliability Diagram). |
| **6. Deployment** | Integrate insights into the CRM lifecycle. | • **Deliverable**: "High-Risk" Customer List for Marketing Squad.<br>• **Artifact**: Serialize model (`joblib`) for batch inference. |

---

### Installs:

In [0]:
%%capture
%pip install -r '../requirements.txt'
# Command to restart the kernel and update the installed libraries
%restart_python

### Imports:

In [0]:
# ======================================================== #
# Standard library
# ======================================================== #
import warnings
import os
import joblib


# ======================================================== #
# Third-party libraries - data analysis and visualization
# ======================================================== #
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# ======================================================== #
# Third-party libraries - modeling and preprocessing
# ======================================================== #
import optuna

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.compose import ColumnTransformer

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

import shap
warnings.filterwarnings('ignore')


In [0]:
# SRC/ Functions Utils:
import sys
sys.path.append('../src')
from visualization import *
from utils import *
from modeling_utils import *

### Load the data

In [0]:
df = pd.read_csv('../data/BankChurners.csv')

### Verify successful load with some randomly selected records


In [0]:
df.sample(9)

In [0]:
df.info()


## 3 - Data Preparation

In [0]:
GraphicsData(data = df).plot_target_analysis(target_col='Attrition_Flag', colors=['#1abc9c', '#ff6b6b'])


### 3.1 - Separating training data from test data.

##### Methodological Note:
---

- Prior to conducting the bivariate analysis, the dataset will be partitioned into **Train** and **Test** sets.  
The primary objective is to prevent **Data Leakage**, ensuring that all statistical insights, outlier treatments, transformation decisions, and feature engineering strategies are derived exclusively from the training data.

> This approach preserves the integrity of the validation process and guarantees that performance metrics reflect true generalization capacity rather than information contamination.

- The `stratify` parameter will be applied within the `train_test_split` procedure.  
Given that churn prediction is a **binary classification problem**, maintaining the original **class proportion** across both subsets is statistically mandatory.

> Stratified sampling preserves the prior probability distribution of the target variable, avoiding distortions in class balance that could bias model training, threshold calibration, and performance evaluation metrics (e.g., Recall, Precision, ROC-AUC).

---


##### Adjusting the target variable
---

I will adjust the target column, `Attrition_Flag`. Since this is a categorical variable with binary classification:

---

- **Existing Customer (Non-churner)**: will receive the **value 0**.

  These are customers who still use the credit card services provided by the bank.

---

- **Attrited Customer (Churner)**: will receive the **value 1**.

---

> This approach will facilitate the calculation of correlation statistics between variables, since the target variable will already be properly encoded.

---



In [0]:
map_churn = {
    'Existing Customer': 0,
    'Attrited Customer': 1
}
df['Attrition_Flag'] = df['Attrition_Flag'].map(map_churn).astype('int8')
df = df.rename(columns={'Attrition_Flag': 'churn'})

X = df.drop(columns = ['churn'])
y = df['churn'].astype('int8').copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.2, 
    stratify = y,  
    random_state = 33
)

In [0]:
# Checking the proportions of the target variable
print(f'Shape of training: {X_train.shape}')
print(f'Shape of test: {X_test.shape}')

print('\n--- Churn Rate (Stratify Validation) ---')
print(f'Original: {y.mean():.2%}')
print(f'Train:    {y_train.mean():.2%}')
print(f'Test:    {y_test.mean():.2%}')

### 3.2 - Construction of Data Preparation Pipelines
---


- For the data preparation stage, **two distinct pipelines** were developed: one designed for **linear models** and the other for **tree-based models**.


> This separation was adopted because each family of algorithms has its own preprocessing requirements, especially with regard to **scaling numerical variables** and **encoding categorical variables**.


- As discussed in the EDA, the initial decision was to remove only the variables **`equip`** and **`wireless`**, since they showed high correlation with **`equipmon`** and **`wiremon`**, respectively.


- The other highly correlated variables were deliberately retained at this initial stage.


> This decision allows the modeling process to empirically evaluate how different algorithms handle **multicollinearity**, **informational redundancy**, and the possible **marginal predictive gain** associated with these variables.


- In both pipelines, the data preparation flow followed the same general structure:


- **Variable type optimization**, with the objective of ensuring structural consistency, reducing memory usage, and adapting the data to the computational requirements of the algorithms.

- **Feature engineering**, with the creation of new derived variables based on findings from the exploratory analysis and domain knowledge.

- **Model-family-specific preprocessing**, respecting the technical particularities of each modeling approach and ensuring compatibility between the transformed data and the algorithms used.


---


### 3.3 - Checking the numeric variables

In [0]:
pipeline_fg = Pipeline(
    steps = [
        ('types_vars', DtypeOptimizer()),
        ('fg', FeatureEngineer())
    ]
)
train_set_fg = pipeline_fg.fit_transform(X_train)
train_set_fg.head()

In [0]:
GraphicsData(train_set_fg.select_dtypes(include = ['float32'])).numerical_histograms()

### 3.4- Pre-processor for Linear Models
---


In [0]:
# Columns
one_hot_cols = [
    'gender', 'education_level', 'marital_status', 'income_category','card_category', 'trans_ct_bucket', 'trans_amt_bucket', 'tenure_bucket'
]

num_cols = [
    'utilization_amont', 'avg_utilization_ratio', 'total_revolving_bal', 'customer_age', 'dependent_count', 'months_on_book',
    'total_relationship_count', 'months_inactive_12_mon','contacts_count_12_mon', 'total_trans_ct'
]
 
skew_cols = [
    'credit_limit', 'total_amt_chng_q4_q1', 'total_trans_amt', 'total_ct_chng_q4_q1','avg_ticket', 'inactive_per_tenure','total_spending', 'change_gap_amt_ct', 
    'credit_revolving'
]

bin_cols = [
    'months_inactive_12_mon_0', 'contacts_count_12_mon_0','dependent_count_0', 'total_revolving_bal_0', 'total_amt_chng_q4_q1_0',
    'total_ct_chng_q4_q1_0', 'avg_utilization_ratio_0','low_activity_low_value_flag', 'is_pos_graduate', 'total_amt_q75','total_ct_q75'
]

drop_cols = [
    'clientnum', 
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_1',
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_2',
    'avg_open_to_buy'
]

# --------------------------------------------------------------------------------------------------------------------------------------#
# Pipelines

# Numerical columns with asymmetry
num_log = Pipeline(
    steps = [
        ('log', FunctionTransformer(np.log1p, feature_names_out = 'one-to-one')),
        ('std', StandardScaler()),
    ]
)

# One Hot Config
one_hot_encoder = OneHotEncoder(
    drop = 'first', 
    sparse_output = False, 
    handle_unknown = 'ignore', 
    feature_name_combiner = 'concat', 
    dtype = np.int8
)

linear_preprocess = ColumnTransformer(
    transformers = [
        ('num_cols', StandardScaler(),num_cols), 
        ('skew_cols', num_log, skew_cols),
        ('one_hot_cols', one_hot_encoder, one_hot_cols),
        ('bin_cols', 'passthrough', bin_cols),
        ('drop_cols', 'drop', drop_cols)
    ],
    remainder = 'passthrough',
    verbose_feature_names_out = False,
).set_output(transform = 'pandas')


linear_pipeline = Pipeline(
    steps = [
       ('types_vars', DtypeOptimizer()), 
       ('feature_eg', FeatureEngineer()),
       ('linear_preprocess', linear_preprocess)
    ]
)

X_train_prepared_linear = linear_pipeline.fit_transform(X_train)

In [0]:
X_train_prepared_linear.head()

### 3.5 - Pre-processor for Tree Models
---

In [0]:

# Columns
one_hot_cols = [
    'gender', 'marital_status', 
]

ord_cols = [
    'education_level', 'income_category', 'card_category', 'trans_ct_bucket', 'trans_amt_bucket', 'tenure_bucket'
]


drop_cols = [
    'clientnum', 
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_1',
    'naive_bayes_classifier_attrition_flag_card_category_contacts_count_12_mon_dependent_count_education_level_months_inactive_12_mon_2',
    'avg_open_to_buy'
    ]


# One Hot Config
one_hot_encoder = OneHotEncoder(
    drop = 'first',
    sparse_output=False,
    handle_unknown = 'ignore',
    feature_name_combiner = 'concat',
    dtype = np.int8
)

# Create Ordinal pipeline

# Create Ordinal order
ordinal_features_order = {
'education_level': ['Uneducated', 'High School', 'College', 'Graduate', 'Post-Graduate', 'Doctorate', ],
'income_category': ['Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +', ],
'card_category': ['Blue', 'Silver', 'Gold', 'Platinum'],
'trans_ct_bucket': ['very_low_activity', 'low_activity', 'medium_activity', 'high_activity', 'very_high_activity'],
'trans_amt_bucket': ['very_low_activity', 'low_activity', 'medium_activity', 'high_activity', 'very_high_activity'], 
'tenure_bucket': ['new', 'developing', 'established', 'loyal']
} 

ordinal_categories = [
    ordinal_features_order['education_level'],
    ordinal_features_order['income_category'],
    ordinal_features_order['card_category'],
    ordinal_features_order['trans_ct_bucket'],
    ordinal_features_order['trans_amt_bucket'],
    ordinal_features_order['tenure_bucket'],
]

ordinal_pipeline = Pipeline(
    steps=[
        (
            'ordinal_encoder',
            OrdinalEncoder(
                categories = ordinal_categories,
                dtype = np.int8,
                handle_unknown='use_encoded_value', 
                unknown_value = -1
            ),
        ),
    ],
)


tree_preprocess = ColumnTransformer(
    transformers=[
        ('one_hot_cols', one_hot_encoder, one_hot_cols),
        ('ord_cols', ordinal_pipeline, ord_cols),
        ('drop_cols', 'drop', drop_cols),
    ],
    remainder = 'passthrough',
    verbose_feature_names_out = False,
).set_output(transform = 'pandas')
      
tree_pipeline = Pipeline(steps=[
    ('types_vars', DtypeOptimizer()),
    ('feature_eg', FeatureEngineer()),
    ('tree_preprocess', tree_preprocess),
])

X_train_prepared_tree = tree_pipeline.fit_transform(X_train)

In [0]:
X_train_prepared_tree.head()

## 4 - Modeling
---

##### Model Evaluation and Selection Strategy
---


- The primary metric defined for this project will be **AUC-ROC**.


> Since the problem involves **imbalanced classes** and the central objective is to develop a model capable of estimating the **probability of churn**, AUC-ROC is appropriate because it provides a comprehensive view of the model's discriminative ability between the two classes.


- As secondary metrics, **F1-score** and **recall for the churn class** will also be monitored.


> The **F1-score** will be used to evaluate the balance between **precision** and **recall**, while **recall for the churn class** will receive special attention, since correctly identifying customers at higher risk of churn is essential for guiding retention actions.


> This choice is strategically relevant, since **retaining a customer** through engagement and loyalty campaigns is approximately **five times less costly** than **acquiring a new customer**.


- Initial training will be conducted using **cross-validation**, with the objective of increasing the robustness of model evaluation at this stage of the project.


> **5 folds** will be used, and the selection of the best model will consider not only the **highest mean AUC-ROC**, but also the **lowest variability** among the results observed across the different folds, seeking consistent performance and greater generalization capability.


- Simpler models, such as **Logistic Regression** and **Decision Tree Classifier**, will be tested.


> This choice prioritizes solutions with **lower complexity** and **greater interpretability**, which becomes especially important given a relatively small dataset with only **200 records**, reducing the risk of **overfitting**.


- And more robust models, such as **Linear SVC** and **LightGBM**, will also be evaluated.


> The objective of this comparison is to verify whether the structure of the data benefits from algorithms with **greater modeling capacity**, making it possible to assess potential performance gains in relation to simpler approaches.


---


### 4.1 - Cross-Validation
---

In [0]:
linear_models = {
    'Logistic Regression': LogisticRegression(),
    'Linear SVC': LinearSVC(),
    'KNeighbors Classifier': KNeighborsClassifier(),
}

eval_linear = classification_kfold_cv(
    models = linear_models,
    X_train = X_train_prepared_linear,
    y_train = y_train,
    n_folds = 5,
    scoring = 'roc_auc'
)
eval_linear

In [0]:
tree_models = {
    'Decision Tree ': DecisionTreeClassifier(),
    'Random Forest ': RandomForestClassifier(),
    'XGB ': XGBClassifier(),
    'LightGBM ': LGBMClassifier(verbosity = -1, ), # verbosity = -1 >> This is just a parameter that disables verbose e in LGBTM.
}
eval_tree = classification_kfold_cv(
    models = tree_models,
    X_train = X_train_prepared_tree,  
    y_train = y_train,
    n_folds = 5,
    
)
eval_tree

In [0]:
df_scores = pd.concat([eval_linear, eval_tree], ignore_index = True)
df_scores = df_scores[['avg_val_score', 'val_score_std', 'model']]


### 4.2 - Models Scores 
---

##### Initial results and direction for feature selection
---

- The results obtained so far can be considered **satisfactory**.

> The **LightGBM** model achieved an **AUC-ROC of 99.34%** and a **variance of 0.0017**, which is a relatively low value, indicating strong stability across the evaluated folds.

- The data in this dataset are the main drivers behind this excellent result, reflecting a high-quality and low-noise database.

> The selected features follow the principle of being extracted or defined before the churn event or before the customer’s continuation with the contracted services.

- Several variables demonstrated good separation ability between **churners** and **non-churners**.

> During the statistical tests performed in the exploratory analysis stage, it was observed that some of these variables, in addition to showing **statistical significance**, also had a relevant impact on the distinction between the groups.

- This behavior was also reflected in the initial training results, especially in the performance of **LightGBM**.

> As a **tree-based** model, its structure tends to handle nonlinear relationships, interactions between variables, and heterogeneous patterns in the data efficiently.

- In light of this scenario, **LightGBM** will be used in the **feature selection** stage.

> The objective will be to **simplify the model** and **remove redundant variables**, with the expectation of further improving predictive performance while also making the solution more **compact** and **interpretable**.
---

In [0]:
models_performance_barplots(data = df_scores.sort_values('avg_val_score'), models_col = 'model')

### 4.3 - Feature Selection 
---

- **RFECV (Recursive Feature Elimination with Cross Validation)** will be used to select the most relevant variables for the model and remove redundant features or those with lower contribution.


> This approach allows the feature selection process to be conducted more robustly, combining recursive feature elimination with cross-validation to identify the configuration that best supports model performance.


- In this way, the goal is to obtain a **simpler**, more **compact**, and more **interpretable** solution, without significant loss in predictive performance.


> The expectation is to reduce the complexity of the feature space, preserve the variables with the strongest informational signal, and improve the efficiency of the modeling process while maintaining consistent performance across evaluations.
---

##### Feature Selection Results with RFE
---

- The results obtained with the **Recursive Feature Eliminator** can be considered **satisfactory**.

> Out of the initial **40 features**, **24 were retained** after the selection process, indicating a reduction in the feature space without losing the most relevant attributes for modeling.

- Among the selected variables, **`total_trans_ct`**, **`total_trans_amt`**, and their derived features stand out.

> This result suggests that factors associated with the **number of transactions** and the **value of those transactions** play an important role in explaining customer churn within the institution.
---

In [0]:
selector = RecursiveFeatureEliminator(
    estimator = LGBMClassifier(verbosity = -1, n_jobs = 1),
    scoring = 'roc_auc',
    n_folds = 5
)

selector.fit(X_train_prepared_tree, y_train)

X_train_selected = selector.transform(X_train_prepared_tree)

print(selector.selected_features_)
print(f'\n\nInitial Features: {len(X_train_prepared_tree.columns)}')
print(f'\nRemainder Features: {len(X_train_selected.columns)}')

In [0]:
X_train_selected.head()

### 4.4 - Hyperparameter Optimization with Optuna
---


- In the **hyperparameter tuning** stage, **Optuna** will be used with the goal of making the hyperparameter optimization process more efficient than exhaustive methods such as **GridSearch**.


> This approach enables a more targeted exploration of the search space, prioritizing hyperparameter combinations with greater performance potential instead of evaluating all possibilities uniformly.


- Instead of testing combinations in a purely random way, **Optuna** uses sampling algorithms such as **TPE (Tree-structured Parzen Estimator)**.


> This mechanism takes into account the history of previous **trials** to identify more promising and less promising regions of the hyperparameter space, making the search process more adaptive and efficient.


- Considering the particular characteristics of this dataset, the definition of the search space was guided by a balance between hyperparameters that help mitigate limitations associated with the **small number of observations**.


> This strategy aims to control model complexity, reducing sensitivity to sample variation and minimizing the risk of excessive fitting to the training data.

---


In [0]:
# Adjusting the weight of negative and positive classes
# scale_pos_weight
ratio = 8500 / 1627  # ---> 5.22

def objective(trial):

    # Custom adjustment for max_depth and num_leaves
    max_depth = trial.suggest_int('max_depth', 3, 8)
    num_leaves = trial.suggest_int('num_leaves', 8, 2 ** max_depth)

    # Params
    params = {
        'scale_pos_weight': ratio,
        'boosting_type': 'gbdt',
        'subsample_freq': 1,
        'objective': 'binary',
        'metric': 'auc',
        'importance_type': 'gain',
        'n_estimators': 2000,
        'verbosity': -1,
        'random_state': 33,
        'n_jobs': 1,

        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log = True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.95),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log = True),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 10.0),

    }

    model = LGBMClassifier(**params)

    cv = StratifiedKFold(
        n_splits = 5,
        shuffle = True,
        random_state = 33
    )

    scores = cross_val_score(
        model,
        X_train_selected,
        y_train,
        scoring = 'roc_auc',
        cv = cv,
        n_jobs = -1
    ) 

    trial.set_user_attr('auc_std', float(np.std(scores)))
    return float(np.mean(scores))


study = optuna.create_study(
    direction='maximize',
    study_name='lgbm_optuna_auc'
)

study.optimize(objective, n_trials = 50)

best_params = study.best_params
best_score = study.best_value
best_std = study.best_trial.user_attrs['auc_std']

print('\nBest Hyperparameters:', best_params)
print('Best Mean ROC-AUC:', best_score)
print('Best ROC-AUC Std:', best_std)

### 4.5 - Final Training 
---

In [0]:
ratio = 8500 / 1627  # ---> 5.22
best_params = {
    'scale_pos_weight': ratio,
    'boosting_type': 'gbdt',
    'subsample_freq': 1,
    'objective': 'binary',
    'metric': 'auc',
    'importance_type': 'gain',
    'n_estimators': 2000,
    'verbosity': -1,
    'random_state': 33,
    'n_jobs': 1,
    'max_depth': 6, 
    'num_leaves': 41, 
    'min_child_samples': 28, 
    'min_child_weight': 0.003766666346845193, 
    'learning_rate': 0.0698776886672484, 
    'subsample': 0.89322138533776, 
    'colsample_bytree': 0.8369071798239733, 
    'min_split_gain': 0.07439179178731464, 
    'reg_alpha': 0.5083828211856398, 
    'reg_lambda': 0.063339361317028, 
    'path_smooth': 4.434880478024902
}

final_model = LGBMClassifier(** best_params)
final_model.fit(X_train_selected, y_train)

## 5 - Evaluation
---

### 5.1 - Testset Preparation
---

In [0]:
X_test_prepared_tree = tree_pipeline.transform(X_test)
X_test_selected =  selector.transform(X_test_prepared_tree)
X_test_selected.head()

### 5.2 - Prediction
---

In [0]:
y_pred = final_model.predict(X_test_selected) 
y_prob = final_model.predict_proba(X_test_selected)[:, 1]

In [0]:
report_df, metrics_df = evaluation_scores(
    y_true = y_test,
    y_score = y_prob,
    threshold = 0.5,
    title_prefix = 'Final Model - '
)


In [0]:
report_df

In [0]:
metrics_df

In [0]:
decile_df = plot_kde_predictions(
    y_true = y_test,
    y_score = y_prob
)

In [0]:
decile_df

### 5.3 - Validation metrics analysis
---

#### Confusion matrix

- On the test data, the model achieved **306 correct predictions out of 325 positive-class cases** and **1669 correct predictions out of 1701 negative-class cases**.

> This result indicates a good balance in the classification of both classes.

> In a churn context, the **false negative** is usually the most expensive type of error, since the cost of acquiring a new customer can be, on average, up to **5 times higher** than the cost of retention.

---

#### Classification metrics
---

##### Precision

- **90.53%**

> For every 100 customers the model classifies as churn, approximately **90 actually churn**, while the other **10 do not cancel**.

> From a business perspective, this means that for every 100 customers classified as churners, the retention team may direct an unnecessary offer to around **10 customers**. Even so, this operational cost tends to be **relatively low** when compared with the **cost of acquiring** a new customer.
---

##### Recall

- **94.15%**

> For every 100 customers who actually churned, the model identifies **94**, leaving **6 customers** undetected.

> The greatest cost impact lies precisely in these missed customers. However, with **94% Recall** and **90% Precision**, the model shows a healthy balance, in which **it prioritizes not missing churners, even while accepting some false alarms**.
---

##### F1-Score

- **92.31%**

> The **F1-Score** represents the harmonic mean between **Precision** and **Recall**, serving as a summary of the balance between the two main types of error.

- Spending resources on customers who are not going to leave (**Precision**).
- Failing to act on customers who are actually going to churn (**Recall**).

> With an **F1 of 92%**, the model maintains a strong balance between both dimensions. If it were optimized only for one of these metrics, the other would likely decline.
---

#### Ranking and probability metrics
---

##### ROC-AUC - Main metric

- **99.55%**

> If one customer who churned and one who stayed are selected at random, there is a **99.55% chance** that the model will assign a **higher churn probability to the customer who actually churned**.

> This shows that the model can separate the two groups almost perfectly. The ROC curve confirms this visually, rising almost vertically and approaching the upper-left corner.

> The business impact is highly positive. If stakeholders need to prioritize retention actions, such as having a budget to contact only 500 customers, the model is likely to rank those with the highest churn risk correctly.
---

##### PR-AUC

- **97.78%**

> **Average Precision** directly measures whether, when the model classifies a customer as churn, it is correct and whether it is also able to identify most actual churners.

> The baseline in this dataset is approximately **16% churners** versus **84% non-churners**. Therefore, a **PR-AUC of 97.78%** indicates that the model is highly effective, especially in identifying churners, and not only in overall classification performance, as reflected by **ROC-AUC**.
---

##### KS (Kolmogorov-Smirnov)

- **93.88%**

> This metric measures the maximum separation between the cumulative distributions of the two classes.

> A model with **KS above 0.80** already demonstrates high separation ability. Therefore, a result of **93.88%** reinforces that the model has excellent discriminatory power.
---

##### Brier Score

- **0.0197**

> The **Brier Score** measures how well-calibrated the predicted probabilities are. In practical terms, it is the mean squared error between the predicted probability and the actual outcome, where **lower is better**.

> Considering the real business impact, if the model indicates that a customer has an **80% probability** of churning, a **low Brier Score** suggests that this probability is reliable.

> This makes it possible to structure personalized strategies, such as:

- **90% probability:** immediate contact from the retention team.
- **Probability between 60% and 90%:** automatic loyalty offer.
- **Probability between 30% and 60%:** active monitoring.
---

#### Decile analysis — risk ranking
---

| Decile | Event Rate | Interpretation |
| --- | --- | --- |
| 1–7 | 0.0% | The model shows extremely high confidence that these customers will stay. |
| 8 | 2.5% | Transition zone, where a few churners begin to appear. |
| 9 | 58.4% | High-risk zone. |
| 10 | 99.5% | Almost all customers are churners. |

> This indicates that the model concentrates virtually all churn in the **last two deciles**, that is, in the **top 20% highest-risk customers**.

> From a business perspective, this means that by targeting only the **20% of customers with the highest scores**, it is possible to capture approximately **99% of all churners**.
---


### 5.4 - SHAP Analysis
---

> In churn prediction projects, **SHAP** is especially useful for two practical reasons: it helps prioritize customers with the highest predicted churn risk and explains why those customers were ranked at the top.

> This is directly aligned with retention operations, since companies usually do not act on the entire customer base, but instead focus on customers with the highest expected risk and business value.

> At a global level, **SHAP** helps identify the main churn drivers across the portfolio, supporting prioritization, communication with business stakeholders, and feature validation.

> At a local level, **SHAP** explains individual predictions, which is useful for understanding specific high-risk customers and designing more targeted retention actions.

> It is important to remember that **SHAP** improves model interpretability, but it does not establish causality. In this project, its role is to complement performance metrics by showing which features are driving churn scores and whether those drivers make business sense.
---

### 5.5 - Individual explanations
---

In [0]:
# Generating the explanatory objects

explainer = shap.Explainer(final_model)
shap_values = explainer(X_test_selected)


### 5.6 - Churn Customer `(f(x) = +13.209)`
---

- **Strong positive prediction** = very high probability of churn. This customer is likely to cancel.

##### Main risk factors
---

- `total_trans_amt` = **40** >>> contribution of **+5.33** *(Low transaction amount, strong signal)*

- `total_trans_ct` = **2472** >>> contribution of **+3.89** *(Low transaction volume)*

- `total_relationship_count` = **2** >>> contribution of **+1.77** *(Few products with the bank)*

- `inactive_per_tenure` = **12** >>> contribution of **+1.39** *(High inactivity)*

- `total_revolving_bal` = **0** >>> contribution of **+1.27** *(Zero revolving balance, customer disengaged from credit usage)*

- `total_ct_chng_q4_q1` = **0.429** >>> contribution of **+1.07** *(Decline in transaction frequency)*

- `change_gap_amt_ct` = **1.165** >>> contribution of **+0.98** *(Low volume in the combined variation pattern)*

##### Profile
---

> Customer with several warning signs: few transactions, low volume, few products, zero revolving balance, and high inactivity. This is a classic profile of progressive disengagement prior to cancellation.
---

In [0]:
# Riskiest customer
top_idx = np.argmax(y_prob)

print('\n-Riskiest customer')
print(f'\n-Position {top_idx}')
print(f'\n-Predicted probability: {y_prob[top_idx]}')

print('\n>>>The data of customer:\n')
print(X_test_selected.iloc[top_idx])

shap.plots.waterfall(shap_values[top_idx])

%md
### 5.7 - Intermediate-Zone Customer `(f(x) = -8.926)`
---

- **Strong negative prediction**, but less extreme than that of retained customers = low probability of churn, although with mixed signals.

##### Protective factors
---

- `total_trans_ct` = **126** >>> **-3.7** *(High transaction volume, strong protective factor)*

- `inactive_per_tenure` = **23** >>> **-1.02** *(Low inactivity relative to tenure)*

- `total_revolving_bal` = **1244** >>> **-0.72** *(Considerable revolving balance maintained by the customer)*

##### Risk factors
---

- `avg_ticket` = **120.44** >>> **+1.69** *(Higher average ticket may indicate concentrated rather than recurring usage)*

- `total_trans_amt` = **15176** >>> **+1.27** *(Higher total transaction amount may indicate a different behavioral pattern)*

- `contacts_count_12_mon` = **3** >>> **+0.41** *(A higher number of contacts with the bank may indicate dissatisfaction or relationship friction)*

##### Profile
---

> Customer with a high transaction frequency, which is the main protective factor, combined with a higher average ticket and higher total spending, along with some signs of friction with the bank. Even so, the transaction volume outweighs the risk factors, keeping this customer in an intermediate zone with a low probability of churn.
---

In [0]:
# Intermediate customer
sorted_idx = np.argsort(y_prob)
mid_idx = sorted_idx[len(sorted_idx) // 2]

print('\n-Intermediate customer')
print(f'\n-Position {mid_idx}')
print(f'\n-Predicted probability: {y_prob[mid_idx]}')

print('\n>>>The data of customer:\n')
print(X_test_selected.iloc[mid_idx])

shap.plots.waterfall(shap_values[mid_idx])


### 5.8 - Retained Customer `(f(x) = -16.209)`
---

- **Strong negative probability** = very low probability of churn. This customer is unlikely to cancel.

##### Main protective factors
---

- `inactive_per_tenure` = **31** >>> contribution of **-2.10** *(Low inactivity relative to tenure)*

- `total_trans_ct` = **81** >>> contribution of **-1.84** *(High transaction frequency)*

- `total_trans_amt` = **3968** >>> contribution of **-1.15** *(Good financial volume)*

- `total_amt_chng_q4_q1` = **0.679** >>> contribution of **-0.96** *(Moderate transaction variation)*

##### Profile
---

> Active customer with a high transaction frequency, solid volume both in transaction count and transaction value, moderate revolving balances, and a low number of contacts with the bank in recent months.
---

In [0]:
# Less risky customer
low_idx = np.argmin(y_prob)

print('\n-Less risky customer')
print(f'\n-Position {low_idx}')
print(f'\n-Predicted probability: {y_prob[low_idx]}')

print('\n>>>The data of customer:\n')
print(X_test_selected.iloc[low_idx])

shap.plots.waterfall(shap_values[low_idx])

### 5.9 - Analysis of the impact of features on the model output.
---

In [0]:
shap.plots.bar(shap_values)

In [0]:
shap.plots.beeswarm(shap_values)

### 5.10 - Impact of variables on model classifications
---

> In this chart, each point represents a customer, positioned by the **SHAP** value on the x-axis and colored according to the feature value (**red = high**, **blue = low**).
---

##### `total_trans_ct` *(importance: +2.4)*
---

- **Most important feature in the model.**

> Customers with fewer transactions are more likely to churn. Higher transaction counts are associated with a lower probability of cancellation.

> **Card usage frequency is the main churn indicator**. When a customer stops transacting, this becomes a signal of possible **exit**.
---

##### `total_trans_amt` *(importance: +1.57)*
---

- **Second most important feature in the model.**

> It shows a different pattern from `total_trans_ct`: there is no equally clear linear relationship between total transaction amount and churn. The red points appear more concentrated in an intermediate zone, without a clear tendency toward either the left or the right.

> **This indicates that not only frequency, but also financial volume matters**. However, transaction volume showed a more dispersed behavior, suggesting that high transaction values do not always act as a protective factor against cancellation.
---

##### `inactive_per_tenure` *(importance: +1.01)*
---

> Customers with higher inactivity relative to relationship length are associated with a greater probability of churn. On the other hand, customers with lower inactivity relative to tenure tend to remain with the service.
---

##### `avg_ticket` *(importance: +0.8)*
---

> Customers with higher average ticket values are associated with a greater probability of cancellation. This helps explain the nonlinear behavior of total transaction value in relation to churn.

> Customers with lower ticket values tend to remain with the service. This suggests that more balanced and recurring spending, combined with good transaction frequency, is associated with a protective factor.
---

##### `total_revolving_bal` *(importance: +0.77)*
---

> Customers with a **zero revolving balance** are associated with higher churn risk, indicating possible disengagement from the credit product.

> Higher **revolving balance** levels, on the other hand, are associated with a protective factor, suggesting greater usage and a stronger connection with the service.
---

##### `total_amt_chng_q4_q1` *(importance: +0.69)*
---

> Lower values are associated with cancellation, while higher values are related to protection and engagement.

> This indicates that customers with an increase in transaction amount variation tend to be more engaged and less likely to churn.
---

##### `total_relationship_count` *(importance: +0.52)*
---

> Lower values are associated with cancellation, while higher values appear as a protective and engagement-related factor.

> This suggests that customers with a greater number of contracted products or services tend to show lower propensity to churn.
---

##### `contacts_count_12_mon` *(importance: +0.48)*
---

> Higher values are associated with cancellation, while lower values appear as a protective factor.

> This indicates that customers who contacted the bank more frequently may be dissatisfied or trying to solve problems. This is a sign of relationship friction, and not necessarily only of disengagement.
---

##### `change_gap_amt_ct` *(importance: +0.46)*
---

> Lower values are associated with cancellation, while higher values appear as a protective and engagement-related factor.

> This suggests that customers with better combined evolution in transaction value and transaction count show lower propensity to churn and a higher level of engagement.
---

### 5.11 - Business Financial Impact of the Model
---

> These metrics validate the robustness of the trained model and its suitability for deployment in retention strategies. The high **AUC-ROC** enables more assertive campaigns focused on customers classified as churners. Considering the **Customer Acquisition Cost (CAC)** and **Lifetime Value (LTV)**, it is possible to build a hypothetical scenario, since no official figures are available for this institution, in order to estimate the costs involved in retention and the potential customer loss.
---

##### Hypothetical cost scenario

- **Cost to acquire a customer:** US\$ 2,500
- **Cost to retain a customer classified as a churner:** US\$ 500

> Based on the confusion matrix:

- **325 churners**
- **1701 non-churners**
- The model classified **338 customers as churners**
---

##### 1. Cost of false positives

> **False positives** represent non-churners who were classified as churners.

- With a **precision of 90.53%**, there were **32 false positives**.
- Cost: **32 × US\$ 500 = US\$ 16,000**
- Total retention campaign cost: **338 × US\$ 500 = US\$ 169,000**
- Share of incorrectly allocated resources: **US\$ 16,000 / US\$ 169,000 = 9.46%**

> This result can be considered satisfactory, since only a small portion of resources would be allocated to customers who would not have left the service anyway. Even so, these actions may still generate a positive impact on customer loyalty.
---

##### 2. Preservation of acquisition investment (CAC)

- Total investment required to replace the **325 churners**: **325 × US\$ 2,500 = US\$ 812,500.00**
- With a **recall of 94.15%**, the model correctly identifies **306 churners**
- Preserved value: **306 × US\$ 2,500 = US\$ 765,000.00**

> Therefore, the model has the potential to preserve up to **US\$ 765,000.00** in acquisition investment, provided that retention actions are effective.

> The high **AUC-ROC** reinforces confidence in making more aggressive decisions with a low risk of wasting resources on customers who are not actually at risk of churn.
---

##### 3. Lifetime Value (LTV) assessment

> **Lifetime Value (LTV)** is a strategic metric that estimates the total value generated by a customer throughout their relationship with the institution.

> This metric helps assess whether investments in **acquisition (CAC)** and **retention** are financially justified.

> The simplified formula for calculating LTV is:

**LTV = Average Ticket × Purchase Frequency × Average Relationship Duration**

> For illustrative purposes, consider the following estimated values based on credit card customers:

- **Average monthly ticket:** US\$ 150 *(estimated value, since no official data is available)*
- **Purchase frequency:** monthly
- **Average relationship duration:** 3 years (36 months), based on the bank’s customer history

**Estimated LTV = 150 × 12 × 3 = US\$ 5,400**

> This value represents the average return the bank can expect from each customer over a three-year period.

> When compared with the **average CAC of US\$ 2,500**, this results in an **LTV/CAC ratio of 2.16**, indicating a healthy and sustainable relationship, since the value generated per customer is more than double the acquisition cost.
---

##### Potential revenue preserved

> This analysis reinforces the importance of effective retention strategies.

- By preserving churners, the model not only avoids losing the acquisition investment but also **protects these customers’ future LTV**.

- **Potential gross revenue** over a 3-year period for customers identified as churners and successfully retained:  
  **306 × US\$ 5,400 = US\$ 1,652,400.00**

- In addition, with a **recall of 94.15%**, the model has the potential to preserve up to **US\$ 765,000.00** in acquisition investment (**CAC**) and **US\$ 1.65 million** in future gross revenue (**LTV**), provided that retention actions are effective.
---

##### Conclusion

> A predictive model with high **AUC-ROC** and high **recall** not only reduces immediate losses but also **maximizes long-term customer value**, directly contributing to the institution’s profitability and sustainability.
---


## 6- Deployment:
---

### 6.1 - Save Model and Pipeline
---

In [0]:
# =========================================================
# FINAL INFERENCE PIPELINE
# =========================================================
inference_pipeline = Pipeline(steps = [
    ('tree_pipeline', tree_pipeline),
    ('selector', selector),
    ('model', final_model)
])

# =========================================================
# SAVE ARTIFACT
# =========================================================
os.makedirs('./artifacts', exist_ok = True)

joblib.dump(inference_pipeline, '../artifacts/churn_pipeline_v1.pkl')

print('✅ Complete churn pipeline saved successfully!')

### 6.2 - Function of Classification
---

In [0]:
def predict_churn(input_data: pd.DataFrame) -> pd.DataFrame:
    """
    Perform end-to-end churn inference from raw input data.
    Returns predicted class and churn probability.
    """
    pipeline_loaded = joblib.load('./artifacts/churn_pipeline_v1.pkl')

    pred_class = pipeline_loaded.predict(input_data)
    pred_proba = pipeline_loaded.predict_proba(input_data)[:, 1]

    result = input_data.copy()
    result['predicted_class'] = pred_class
    result['churn_probability'] = pred_proba

    return result

In [0]:
# =========================================================
# USER ACCEPTANCE TEST
# =========================================================
sample_input = X_test.head(20).copy()

try:
    predictions = predict_churn(sample_input)

    print('--- INFERENCE REPORT ---')
    print(predictions[['predicted_class', 'churn_probability']])

except Exception as e:
    print(f'Deployment error: {e}')

In [0]:
def risk_band(p: float) -> str:
    if p >= 0.80:
        return 'very_high'
    elif p >= 0.60:
        return 'high'
    elif p >= 0.40:
        return 'medium'
    elif p >= 0.20:
        return 'low'
    return 'very_low'

In [0]:
predictions['risk_band'] = predictions['churn_probability'].apply(risk_band)
predictions['risk_band']

## Conclusion

> The main objective was achieved. On the test data, the model reached **99.55% AUC-ROC**, showing its ability to generate effective probabilities that are highly consistent with the observed data. This makes it possible to establish strategies based on ranking customers with the highest probabilities of **churn**.

> The model demonstrates strong **financial potential for retention initiatives**, as it combines high recall with good precision, allowing the institution to identify most **at-risk customers** while also limiting wasted retention efforts.

> In a hypothetical scenario, its use could help preserve both the **acquisition investment** and a significant share of customers' future value, making it a strategically relevant tool to support more efficient, data-driven retention decisions.